# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Randaadad/FlyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("Connected to Hugging Face warehouse.")

Connected to Hugging Face warehouse.


## 1. Unit of analysis + time window

Unit of analysis: One row represents the daily performance of one content page for one pseudonymized client on one report date.

Time window: I will use March 2026 as the development window. I will not use the final June 2026 _sample as a development window because it represents the latest month and may overlap with future outcomes.

In [12]:
q1 = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT content_hash_id) AS content_pages,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT report_date) AS report_dates,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

display(con.sql(q1).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,content_pages,clients,report_dates,first_date,last_date
0,9841378,331437,55,31,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

Features: impressions, clicks, CTR, average position, sessions. These are observed page-level signals that can be used to rank pages.

Label / proxy: CTR opportunity. I will treat a page with enough impressions and CTR below the expected CTR for its position tier as a directional opportunity proxy.

Context: content type, freshness, client, and report date. These provide context for interpreting differences between pages.

Excluded: future outcome fields and any label-derived fields are excluded from the feature set because they would not be available at the decision moment and could cause leakage.

In [11]:
schema = con.sql("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
q_grain = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(
        CAST(client_id AS VARCHAR), '|',
        CAST(content_id AS VARCHAR), '|',
        CAST(report_date AS VARCHAR)
    )) AS unique_client_page_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

display(con.sql(q_grain).df())


In [ ]:
q_window = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT content_id) AS content_pages,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

display(con.sql(q_window).df())

In [ ]:
q_missing = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE impressions_available IS TRUE
    ) AS impressions_available,
    COUNT(*) FILTER (
        WHERE ctr_available IS TRUE
    ) AS ctr_available
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

display(con.sql(q_missing).df())

## 4. Data limits
Data limits: This data can show observed search and engagement performance, but it cannot prove that changing a page caused CTR or engagement to improve. The history may also be unbalanced across pages, and some early rows may contain GSC-only information. Time windows can overlap when creating past-to-future labels, so future outcome periods must be kept separate from decision-time features. The final _sample month should be treated as a sealed test window rather than used to develop the label or features.

In [ ]:
print("Development window: 2026-03")
print("Final _sample month should remain sealed for development.")
print("Future outcome information must not be used as a feature.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.